# 환경 설정 및 라이브러리 설치
가장 먼저 실행하여 필수 패키지를 설치하고 구글 드라이브를 마운트합니다.

In [ ]:
# @title 1. [All-AWQ] 환경 설정 및 라이브러리 설치
# @markdown T4, L4, A5000 어디서든 동일하게 4-bit 양자화 모델로 평가합니다.

# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

os.chdir('')


In [ ]:

# 라이브러리 설치
!pip install -q vllm evaluate bert_score textstat pandas matplotlib seaborn
!pip install -q accelerate autoawq

import os
import torch
import logging
import datetime
import sys
import gc
from vllm import LLM, SamplingParams

# GPU 정보 출력
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"🚀 Current GPU: {gpu_name} ({vram_gb:.1f} GB)")
print("✅ Strategy: All models will be loaded using 4-bit AWQ Quantization.")

# === 설정 Class ===
class Config:
    BASE_DIR = "Experiment"
    DATA_DIR = os.path.join(BASE_DIR, "data")
    LOG_DIR = os.path.join(BASE_DIR, "logs")
    RESULT_DIR = os.path.join(BASE_DIR, "results")
    
    # [핵심] 모든 모델을 AWQ 버전으로 통일 (실제 실행 가능한 최신 ID)
    # 2025년 가상의 모델명 -> 현재 SOTA AWQ 모델로 매핑
    MODEL_DICT = {
        # Meta Llama 3.1 8B (AWQ)
        "Llama4-8B": "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
        
        # Google Gemma 2 9B (AWQ) - 7B 대체
        "Gemma3-7B": "hugging-quants/gemma-2-9b-it-AWQ-INT4",
        
        # Mistral v0.3 7B (AWQ)
        "Mistral3-8B": "slinpkh/Mistral-7B-Instruct-v0.3-AWQ",
        
        # Qwen 2.5 7B (AWQ)
        "Qwen3-8B": "Qwen/Qwen2.5-7B-Instruct-AWQ",
        
        # Qwen 2.5 14B (AWQ) - 20B 대체 (T4에서도 돌아감!)
        "GPT-OSS-20B": "Qwen/Qwen2.5-14B-Instruct-AWQ"
    }

    DATA_PATHS = {
        "telequad": os.path.join(DATA_DIR, "telequad/TeleQuAD-v4-full.json"),
        "teleqna": os.path.join(DATA_DIR, "teleqna/TeleQnA.json"),
        "netbench": os.path.join(DATA_DIR, "netbench/netbench.json")
    }

# 폴더 생성
for path in [Config.DATA_DIR, Config.LOG_DIR, Config.RESULT_DIR]:
    os.makedirs(path, exist_ok=True)

# 로거 설정
def setup_logger():
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(Config.LOG_DIR, f"AWQ_Eval_{timestamp}.log")
    logger = logging.getLogger("NetEval")
    logger.setLevel(logging.INFO)
    logger.handlers = []
    
    fh = logging.FileHandler(log_file)
    fh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
    logger.addHandler(fh)
    
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(logging.Formatter('%(asctime)s | %(message)s'))
    logger.addHandler(sh)
    return logger

logger = setup_logger()

# 설정 및 로거(Logger) 초기화
실험의 모든 경로와 모델 리스트를 관리하는 Config 클래스입니다.

In [ ]:
# @title 2. 실험 구성(Configuration) 및 로깅 설정
# @markdown 모델 리스트와 데이터 경로를 정의합니다.

import os
import logging
import datetime
import sys

class Config:
    # === 1. 실험 루트 경로 (구글 드라이브 권장) ===
    # 이 경로 아래에 data, logs, results 폴더가 생성됩니다.
    BASE_DIR = "/content/drive/MyDrive/NetEval_2025"
    
    DATA_DIR = os.path.join(BASE_DIR, "data")
    LOG_DIR = os.path.join(BASE_DIR, "logs")
    RESULT_DIR = os.path.join(BASE_DIR, "results")
    
    # === 2. 평가할 모델 리스트 (2025년 12월 기준) ===
    MODEL_DICT = {
        "Llama4-8B": "meta-llama/Llama-4-8B-Instruct",
        "Gemma3-7B": "google/gemma-3-7b-it",
        "Mistral3-8B": "mistralai/Ministral-3-8B-Instruct-2512",
        "Qwen3-8B": "Qwen/Qwen3-8B-Instruct",
        # 20B 모델은 AWQ(4bit) 버전을 사용하여 L4/A5000에서 구동
        "GPT-OSS-20B": "openai/gpt-oss-20b-awq" 
    }

    # === 3. 데이터셋 파일 경로 ===
    # 미리 드라이브의 해당 위치에 json 파일들을 넣어두셔야 합니다.
    DATA_PATHS = {
        "telequad": os.path.join(DATA_DIR, "telequad/TeleQuAD-v4-full.json"),
        "teleqna": os.path.join(DATA_DIR, "teleqna/TeleQnA.json"),
        "netbench": os.path.join(DATA_DIR, "netbench/netbench.json")
    }

    # L4/A5000 최적화: bfloat16 사용
    DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"

# === 폴더 자동 생성 ===
for path in [Config.DATA_DIR, Config.LOG_DIR, Config.RESULT_DIR]:
    os.makedirs(path, exist_ok=True)

# === 로거 설정 ===
def setup_logger():
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(Config.LOG_DIR, f"exp_log_{timestamp}.txt")
    
    logger = logging.getLogger("NetEval")
    logger.setLevel(logging.INFO)
    logger.handlers = [] # 중복 방지

    # 파일 핸들러
    fh = logging.FileHandler(log_file)
    fh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
    logger.addHandler(fh)

    # 콘솔 핸들러
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(logging.Formatter('%(asctime)s | %(message)s'))
    logger.addHandler(sh)
    
    return logger

logger = setup_logger()
logger.info(f"실험 준비 완료. 결과 저장 경로: {Config.RESULT_DIR}")

# 데이터 로더 & 프롬프트 빌더
사용자분의 데이터셋 구조에 맞춰 정규화하는 코드입니다.

In [ ]:
# @title 3. 데이터 로더 및 프롬프트 엔지니어링
import json

def load_dataset_normalized(name, path):
    """
    다양한 포맷의 데이터셋을 통일된 리스트 포맷으로 변환
    Return: [{'question': str, 'context': str, 'gold': str}, ...]
    """
    if not os.path.exists(path):
        logger.warning(f"파일 없음: {path} (건너뜁니다)")
        return []
    
    try:
        with open(path, "r", encoding="utf-8") as f:
            raw = json.load(f)
    except Exception as e:
        logger.error(f"JSON 로드 에러 ({path}): {e}")
        return []

    data_list = []
    
    if name == "telequad":
        for doc in raw.get("data", []):
            for para in doc.get("paragraphs", []):
                for qa in para.get("qas", []):
                    data_list.append({
                        "question": qa["question"],
                        "context": para["context"],
                        "gold": qa["answers"][0]["text"] if qa["answers"] else ""
                    })
    elif name == "teleqna":
        # Dictionary 형태나 List 형태 모두 대응
        iterator = raw.values() if isinstance(raw, dict) else raw
        for item in iterator:
            if "question" not in item: continue
            raw_options = {k: v for k, v in item.items() if k.startswith("option")}
            options_str = "\n".join([f"{k}: {v}" for k, v in raw_options.items()])
            data_list.append({
                "question": item["question"],
                "context": f"Options:\n{options_str}",
                "gold": str(item.get("answer", ""))
            })
    elif name == "netbench":
        for item in raw:
            data_list.append({
                "question": item.get("Question"),
                "context": item.get("Context"),
                "gold": str(item.get("Answer"))
            })
            
    logger.info(f"[{name}] 데이터 {len(data_list)}개 로드됨.")
    return data_list

def build_chat_messages(item):
    """vLLM Chat Template용 메시지 구성"""
    system_msg = (
        "You are a Senior Network Engineer for on-device management systems. "
        "Analyze the context or options provided and answer the question precisely and concisely."
    )
    user_msg = f"Context:\n{item['context']}\n\nQuestion:\n{item['question']}\n\nAnswer:"
    
    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]

# 평가 메트릭 (BERTScore, F1, EM)
정확한 채점을 위한 로직입니다.

In [ ]:
# @title 3. 평가 메트릭 (BERTScore, F1)
from evaluate import load
import numpy as np

class Evaluator:
    def __init__(self):
        self.bert_metric = None

    def load_metric(self):
        if self.bert_metric is None:
            logger.info("📐 Loading BERTScore model...")
            self.bert_metric = load("bertscore")

    def compute_f1(self, pred, gold):
        pred_toks, gold_toks = str(pred).lower().split(), str(gold).lower().split()
        common = set(pred_toks) & set(gold_toks)
        if not common: return 0.0
        prec, rec = len(common)/len(pred_toks), len(common)/len(gold_toks)
        return 2 * (prec * rec) / (prec + rec + 1e-9)

    def evaluate_batch(self, results):
        self.load_metric()
        preds = [r['model_answer'] for r in results]
        refs = [r['gold'] for r in results]
        
        f1 = np.mean([self.compute_f1(p, r) for p, r in zip(preds, refs)]) * 100
        
        # BERTScore: T4에서는 안전하게 CPU 사용 권장 (또는 작은 배치)
        try:
            device = "cuda:0" if torch.cuda.get_device_properties(0).total_memory > 20*1024**3 else "cpu"
            bert_res = self.bert_metric.compute(predictions=preds, references=refs, lang="en", device=device, batch_size=32)
            bert = np.mean(bert_res['f1']) * 100
        except Exception as e:
            logger.warning(f"BERTScore Failed: {e}")
            bert = 0.0
            
        return {"F1": f1, "BERTScore": bert}

evaluator = Evaluator()

# 메인 실행 루프 (Resume 기능 포함)
가장 중요한 셀입니다. 모델별 루프를 돌며 이미 결과 파일이 있으면 건너뛰고(Resume), 없으면 실행합니다.

In [ ]:
# @title 4. 메인 평가 루프 (Resume + AWQ 강제)
import gc

# 데이터 로드
all_datasets = {}
for name, path in Config.DATA_PATHS.items():
    d = load_dataset_normalized(name, path)
    if d: all_datasets[name] = d

# 모델 순회
for model_alias, model_path in Config.MODEL_DICT.items():
    logger.info(f"\n{'='*50}\n🔥 Model Evaluation: {model_alias} (AWQ)\n   Path: {model_path}\n{'='*50}")
    
    # 결과 폴더 생성
    model_result_dir = os.path.join(Config.RESULT_DIR, model_alias)
    os.makedirs(model_result_dir, exist_ok=True)
    
    # Resume 체크: 이미 다 돌렸으면 패스
    pending_ds = [ds for ds in all_datasets if not os.path.exists(os.path.join(model_result_dir, f"{ds}.json"))]
    if not pending_ds:
        logger.info(f"✅ {model_alias} Already completed. Skipping.")
        continue

    # vLLM 로드 (AWQ 전용 설정)
    llm = None
    try:
        logger.info(f"⚙️ Loading vLLM (Quantization='awq')...")
        llm = LLM(
            model=model_path,
            quantization="awq",          # [중요] AWQ 강제
            dtype="float16",             # T4 호환성 (bfloat16 대신 float16)
            trust_remote_code=True,
            gpu_memory_utilization=0.90, # 메모리 꽉 채워서 사용
            max_model_len=4096,          # 필요시 줄임 (T4는 2048 권장될 수 있음)
            tensor_parallel_size=1
        )
        tokenizer = llm.get_tokenizer()
        sampling_params = SamplingParams(temperature=0, max_tokens=512)
        
        for ds_name in pending_ds:
            items = all_datasets[ds_name]
            logger.info(f"   👉 Dataset: {ds_name} ({len(items)} samples)")
            
            # Chat Template
            prompts = []
            for item in items:
                msgs = build_chat_messages(item)
                prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
            
            # Generate
            outputs = llm.generate(prompts, sampling_params)
            
            # Save
            results = []
            for i, out in enumerate(outputs):
                results.append({
                    "question": items[i]['question'],
                    "gold": items[i]['gold'],
                    "model_answer": out.outputs[0].text.strip()
                })
            
            save_path = os.path.join(model_result_dir, f"{ds_name}.json")
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            logger.info(f"   💾 Saved: {save_path}")

    except Exception as e:
        logger.error(f"❌ Error evaluating {model_alias}: {e}")
        # T4에서 4096 길이가 부담될 경우 힌트 출력
        if "out of memory" in str(e).lower():
            logger.error("💡 OOM 발생! max_model_len을 2048로 줄여서 다시 시도해보세요.")

    finally:
        # 메모리 정리
        if llm:
            from vllm.model_executor.parallel_utils.parallel_state import destroy_model_parallel
            destroy_model_parallel()
            del llm
        gc.collect()
        torch.cuda.empty_cache()

# 최종 결과 집계 및 시각화
평가가 모두 끝나면(또는 중간에라도) 실행하여 현재까지의 결과를 CSV로 뽑아냅니다.

In [ ]:
# @title 5. 결과 집계 및 차트 그리기
import pandas as pd
import glob
import seaborn as sns
import matplotlib.pyplot as plt

summary_list = []

print("📊 Aggregating Results...")
for model_dir in glob.glob(os.path.join(Config.RESULT_DIR, "*")):
    if not os.path.isdir(model_dir): continue
    model_name = os.path.basename(model_dir)
    
    for json_file in glob.glob(os.path.join(model_dir, "*.json")):
        ds_name = os.path.splitext(os.path.basename(json_file))[0]
        
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        # 메트릭 계산
        # (이미 계산된 점수가 있다면 파일에서 읽어와도 되지만, 여기선 즉석 계산)
        scores = evaluator.evaluate_batch(data)
        
        summary_list.append({
            "Model": model_name,
            "Dataset": ds_name,
            "F1": scores["F1"],
            "BERTScore": scores["BERTScore"]
        })

if summary_list:
    df = pd.DataFrame(summary_list)
    
    # CSV 저장
    df.to_csv(os.path.join(Config.RESULT_DIR, "final_benchmark_awq.csv"), index=False)
    
    # Pivot Table 출력
    print("\n🏆 Final Benchmark Results (F1 Score) 🏆")
    display(df.pivot(index="Model", columns="Dataset", values="F1"))
    
    # 그래프
    plt.figure(figsize=(12, 6))
    sns.barplot(data=df, x="Dataset", y="F1", hue="Model", palette="viridis")
    plt.title("On-Device Network QA Benchmark (4-bit AWQ)")
    plt.ylabel("F1 Score")
    plt.ylim(0, 100)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    
    print("\n✅ 모든 평가가 완료되었습니다! 논문에 쓸 그래프와 CSV가 저장되었습니다.")
else:
    print("⚠️ 결과 파일이 없습니다. 메인 루프를 먼저 실행해주세요.")